# **Project 2: Model Comparison (Phase 2, Forest Cover Type Classification)**

David Almona \
DSC 340: Applied Machine Learning \
Spring 2026

---

**Dataset:** Forest Cover Type (30,000 rows, 54 features, 7-class classification) 

**Metric:** Macro F1 score

**Task:**  Predict the forest cover type (one of 7 tree species) from 54 features: 10 continuous variables and 44 binary indicators.



**Sections:**

01. Import Libraries
02. Data Loading and Brief EDA
03. Train/Test Split
04. Five-Model Pipeline
05. Results Table
06. Verdict
07. References and Acknowledgments

## 01. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import warnings

from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

warnings.filterwarnings('ignore')

## 02. Load Data & Exploratory Data Analysis

- ~~Class distribution (bar chart with percentages).~~ Discuss the imbalance.
- Feature characterization: describe the three feature groups (continuous, wilderness, soil). Identify sparse soil types.
- At least one interaction analysis (e.g., how does elevation distribution differ across
wilderness areas or cover types?).
- ~~Correlation matrix for continuous features.~~ Flag any strong correlations.

In [ ]:
df = pd.read_csv('covtype_30k.csv')
df.head()

In [ ]:
# Check dataset dimensions and data types
print(f"Shape: {df.shape}")
print()
print(f"Data types:\n{df.dtypes}")

In [ ]:
df.describe().round(2)

In [ ]:
# class distribution
target_col = df.columns[-1]
class_counts = df[target_col].value_counts().sort_index(ascending=True)

print("Class Distribution (Raw):\n", class_counts)
print()

class_counts = df[target_col].value_counts(normalize = True).sort_index(ascending=False)


print("Class Distribution (Proportion):")
class_counts.plot(kind = 'barh')
plt.title('Class Distribution')
plt.ylabel('Cover Type')
plt.xlabel('Proportion')
plt.xticks(rotation = 0) 
plt.tight_layout()
plt.show()

# discuss the imbalance:
# Cover_Type 1 & 2 significantly more than the others; Cover_Type 4 have the lowest in number
# It is important to remember to use stratified sampling when running classifiers, or else the classifiers will not perform well.

In [ ]:
## correlation matrix for continuous features
# Soil_Type15 contains all zero so it gets filtered out, manually adding it back

binary_cols = [col for col in df.columns if df[col].nunique() == 2]
continuous_cols = [col for col in df.columns if col not in binary_cols]

binary_df = df[binary_cols + ["Soil_Type15"]]
continuous_df = df.drop(columns = [*binary_cols, "Soil_Type15"])

matrix = continuous_df.drop(columns = target_col).corr()
mask = np.triu(np.ones_like(matrix, dtype = bool), k = 1)

plt.figure(figsize=(8,6))
sns.heatmap(matrix, mask = mask, annot = True, cmap = "coolwarm_r", fmt = ".2f", linewidths = 0.5)
plt.title("Correlation Heatmap")
plt.show()

# observations:
# Hillside_9am-Hillside_3pm strong negative correlation
# Aspect-Hillside_3pm strong positive correlation
# ...
# ...

In [ ]:
## Pairplot
sns.pairplot(continuous_df.drop(columns = target_col), corner = True)

# observations:
# complements the correlation heatmap above
# interesting non linear relationships between `Aspect` and the three `Hillside_` variables
# distance to hydrology variables are heavily skewed towards zero

In [ ]:
## check distribution of Soil_Type

soil_type_df = binary_df.filter(like = 'Soil_Type')
soil_proportions = soil_type_df.mean().round(4).sort_values(ascending=False)*100

prop_table = soil_proportions.reset_index()
prop_table.columns = ['Soil_Type', 'Percentage']

print(prop_table)

# observations:
# Soil_Type29 is ~20% of our data
# 21 Soil_Types with less than 1% (sparse, very very sparse)
# Soil_Type15 with 0.00% (all zeros)

In [ ]:
## check distribution of Wilderness_Area

wilderness_area_df = binary_df.filter(like = 'Wilderness_Area')
wilderness_area_proportions = wilderness_area_df.mean().round(4).sort_values(ascending=False)*100

prop_table = wilderness_area_proportions.reset_index()
prop_table.columns = ['Wilderness_Area', 'Percentage']

print(prop_table)

In [ ]:
## how does elevation distribution differ across wilderness areas

# create a single 'Wilderness_Type' column from the 4 binary columns
wilderness_cols = [col for col in df.columns if 'Wilderness_Area' in col]
df['Wilderness_Type'] = df[wilderness_cols].idxmax(axis=1)

# KDE Plot
plt.figure(figsize=(10, 6))
sns.kdeplot(data = df, x = 'Elevation', hue = 'Wilderness_Type', fill = True, common_norm = False, palette = 'colorblind')
plt.title('Elevation Distribution by Wilderness Area')
plt.xlabel('Elevation (meters)')
plt.ylabel('Density')
plt.show()

# observation:
# Wilderness_Area4: occupies the low elevations, primarily sitting below 2,500 meters.
# Wilderness_Area2: Concentrated at the highest elevations, with its main peak reaching above 3,200 meters.
# Wilderness_Areas 1 & 3: Show overlap across the middle range.

In [ ]:
## how does elevation distribution differ across cover types

plt.figure(figsize=(14, 10))
sns.boxplot(data = df, x = 'Cover_Type', y = 'Elevation')
plt.title('Elevation Distribution by Cover Type')
plt.xlabel('Cover Type')
plt.show()

# observation:
# Cover Type 7: Occupies the highest elevations, typically above 3,300 meters.
# Cover Type 4: Occupies the lowest elevations, typically below 2,300 meters.
# Cover Types 1 & 2 overlap in the high range, while 3 & 6 overlap in the lower-mid range.

## 03. Train/test split

80/20 stratified split, `random_state=42`

In [ ]:
y = df["Cover_Type"]
X = df.drop(columns = ["Cover_Type", "Wilderness_Type"])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, stratify = y, random_state = 42)

print(f'Training: {X_train.shape[0]:,}  Test: {X_test.shape[0]:,}')
print(f'Features: {X_train.shape[1]}')

## 04. Five-model pipeline

Define a `Pipeline` (scaler + model) and a hyperparameter grid for each of the five classifiers. 

Run `GridSearchCV` with `scoring='f1_macro'` and 5-fold `StratifiedKFold`.

Report the best CV macro F1 for each model.

In [ ]:
# define Pipeline and hyperparameter grid for each classifier
# log reg, knn, random forest, svm-rbf, svm-linear

pipelines = {
    'Logistic Regression' : {
        'pipe' : Pipeline([
            ('scaler', StandardScaler()),
            ('clf', LogisticRegression(max_iter = 2000, random_state = 42))
        ]),
        'params' : {
            'clf__C' : [0.01, 0.1, 1, 10, 100]
        }
    },
    'KNN' : {
        'pipe' : Pipeline([
            ('scaler', StandardScaler()),
            ('clf', KNeighborsClassifier())
        ]),
        'params' : {
            'clf__n_neighbors' : [3, 5, 11, 21, 41],
            'clf__weights' : ['uniform', 'distance']
        }
    },
    'Random Forest' : {
        'pipe' : Pipeline([
            ('scaler', StandardScaler()),
            ('clf', RandomForestClassifier(random_state = 42))
        ]),
        'params' : {
            'clf__n_estimators' : [100, 300],
            'clf__max_depth' : [None, 10, 20],
            'clf__max_features' : ['sqrt', 'log2']
        }
    },
    'SVM (RBF)' : {
        'pipe' : Pipeline([
            ('scaler', StandardScaler()),
            ('clf', SVC(kernel = 'rbf', random_state = 42))
        ]),
        'params' : {
            'clf__C' : [0.1, 1, 10, 100],
            'clf__gamma' : ['scale', 'auto', 0.01, 0.1]
        }
    },
    'SVM (Linear)' : {
        'pipe' : Pipeline([
            ('scaler', StandardScaler()),
            ('clf', SVC(kernel = 'linear', random_state = 42))
        ]),
        'params' : {
            'clf__C' : [0.001, 0.01, 0.1, 1, 10]
        }
    }
}

In [ ]:
# run grid search with 5-fold stratified CV, scoring on F1
cv = StratifiedKFold(n_splits = 5, shuffle = True, random_state = 42)

In [ ]:
# Run all five grid searches
results = []

for name, spec in pipelines.items():
    print(f'Running {name}...', end=' ')
    gs = GridSearchCV(
        spec['pipe'], spec['params'], cv=cv,
        scoring='f1_macro', n_jobs=-1, refit=True
    )
    start = time.time()
    gs.fit(X_train, y_train)
    elapsed = time.time() - start
    
    results.append({
        'Model': name,
        'Best CV macro F1': gs.best_score_,
        'Best Params': gs.best_params_,
        'Fit Time (s)': round(elapsed, 1),
        'grid_search': gs  # keep the fitted object for later
    })
    print(f'F1={gs.best_score_:.4f}  ({elapsed:.1f}s)')

print()
print('Done.')

## 05. Results Summary
A summary table showing, for each model: best CV macro F1, best hyperparameters, and training time.

A horizontal bar chart comparing macro F1 across models.

In [ ]:
# Results table (without the grid_search objects)
results_df = pd.DataFrame(results).drop(columns='grid_search')
results_df = results_df.sort_values('Best CV macros F1', ascending=False)
results_df.index = range(1, len(results_df) + 1)
results_df.index.name = 'Rank'
results_df

references:

- Google's Gemini
- https://www.geeksforgeeks.org/data-science/create-a-correlation-matrix-using-python/